# OTT 이탈 예측 — final_merged 기본 vs v2 파생변수 비교

CatBoost Optuna 100 trials | 기본 final_merged vs 파생변수 추가본

In [ ]:
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("라이브러리 로드 완료")

In [ ]:
# integrated_analysis.db에서 membership + user_mapping 조인
mem = pd.read_csv("../../_data/01_raw/Membership.csv", encoding="utf-8-sig")
um  = pd.read_csv("../../_data/01_raw/User_Mapping.csv", encoding="utf-8-sig")

# repurchase 타겟 변환
mem["repurchase"] = (mem["repurchase"] == "O").astype(int)
mem["promotion_yn"]       = (mem["promotion_yn"] == "O").astype(int)
mem["is_churn_prevented"] = (mem["is_churn_prevented"] == "O").astype(int)
mem["is_user_verified"]   = (mem["is_user_verified"] == "Y").astype(int)
mem["gender"] = mem["gender"].map({"F":0,"M":1}).fillna(0).astype(int)

# membership + user_mapping 조인 (uid = user_no)
df = mem.merge(um.rename(columns={"uid":"user_no"}), on="user_no", how="left")

# 불필요 컬럼 제거
drop_cols = ["user_no","product_cd","reg_date","end_date","USER_ID","repurchase_label","registerday"]
df_base = df.drop(columns=[c for c in drop_cols if c in df.columns])

print(f"final_merged shape: {df.shape}")
print(f"기본 피처 수 (타겟 제외): {df_base.shape[1]-1}")
print(f"재결제율: {df_base.repurchase.mean():.3f}")

In [ ]:
df_v2 = df.copy()

# duration_days
df_v2["reg_date_dt"] = pd.to_datetime(df["reg_date"])
df_v2["end_date_dt"] = pd.to_datetime(df["end_date"])
df_v2["duration_days"] = (df_v2["end_date_dt"] - df_v2["reg_date_dt"]).dt.days.clip(0)

# plan_tier, currency_type
tier_map = {
    "pk_1487":"basic","pk_1488":"standard","pk_1489":"premium",
    "pk_2025":"basic","pk_2026":"standard","pk_2027":"premium",
    "pk_1508":"basic","pk_1506":"standard","pk_1507":"premium",
}
df_v2["plan_tier"]     = df_v2["product_cd"].map(tier_map).fillna("기타")
df_v2["currency_type"] = df_v2["product_cd"].isin(["pk_1508","pk_1506","pk_1507"]).map({True:"USD",False:"KRW"})

# 금액 더미
amount_norm = df_v2["amount"].copy()
df_v2["is_promotional_price"] = (amount_norm == 100).astype(int)
df_v2["amt_100"]   = (amount_norm == 100).astype(int)
df_v2["amt_7900"]  = (amount_norm == 7900).astype(int)
df_v2["amt_10900"] = (amount_norm == 10900).astype(int)
df_v2["amt_13900"] = (amount_norm == 13900).astype(int)

# 가입 시간 파생
df_v2["reg_weekday"]    = pd.to_datetime(df_v2["reg_date"]).dt.weekday
df_v2["is_night_signup"]= df_v2["reg_hour"].isin([22,23,0,1,2,3,4,5]).astype(int)
df_v2["is_same_day_cancel"] = (df_v2["duration_days"] == 0).astype(int)
df_v2["age_group"]      = (df_v2["age"] // 10) * 10

# 시청 파생 (user_mapping 컬럼 활용)
df_v2["has_watch_history"]    = (df_v2["view_count"] > 0).astype(int)
df_v2["avg_duration"]         = (df_v2["total_duration"] / (df_v2["view_count"] + 1)).round(2)
df_v2["watch_density"]        = df_v2["view_count"] / (df_v2["duration_days"] + 1)
df_v2["binge_score"]          = df_v2["dur_w1"] / (df_v2["total_duration"] + 1)
df_v2["recency"]              = (df_v2["dur_w3"] == 0).astype(int) * df_v2["duration_days"]
df_v2["stream_watch_interaction"] = df_v2["concurrent_streams"] * df_v2["view_count"]

drop_cols2 = ["user_no","product_cd","reg_date","end_date",
              "reg_date_dt","end_date_dt","USER_ID","repurchase_label","registerday"]
df_feat = df_v2.drop(columns=[c for c in drop_cols2 if c in df_v2.columns])

print(f"v2 파생변수 추가본 피처 수 (타겟 제외): {df_feat.shape[1]-1}")
new_feats = ["duration_days","plan_tier","currency_type","is_promotional_price",
             "amt_100","amt_7900","amt_10900","amt_13900","reg_weekday",
             "is_night_signup","is_same_day_cancel","age_group","has_watch_history",
             "avg_duration","watch_density","binge_score","recency","stream_watch_interaction"]
print(f"추가된 파생변수: {len(new_feats)}개")

In [ ]:
def make_Xy(df):
    feat = [c for c in df.columns if c != "repurchase"]
    X = df[feat].copy()
    for c in X.select_dtypes(include="object").columns:
        X[c] = LabelEncoder().fit_transform(X[c].astype(str))
    X = X.fillna(0)
    return X, df["repurchase"], feat

X_base, y_base, feat_base = make_Xy(df_base)
X_feat, y_feat, feat_feat = make_Xy(df_feat)

Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X_base, y_base, test_size=0.2, random_state=42, stratify=y_base)
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(X_feat, y_feat, test_size=0.2, random_state=42, stratify=y_feat)

print(f"기본:    피처 {len(feat_base)}개 | Train {len(Xb_tr)} / Test {len(Xb_te)}")
print(f"파생추가: 피처 {len(feat_feat)}개 | Train {len(Xf_tr)} / Test {len(Xf_te)}")

In [ ]:
N_TRIALS = 100

def obj_cat(trial, X_tr, y_tr):
    p = dict(
        iterations    = trial.suggest_int("iterations", 200, 800),
        learning_rate = trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        depth         = trial.suggest_int("depth", 3, 9),
        l2_leaf_reg   = trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        random_seed=42, verbose=0,
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(CatBoostClassifier(**p), X_tr, y_tr,
                           cv=cv, scoring="roc_auc", n_jobs=-1).mean()

def run_cat(X_tr, y_tr, X_te, y_te, label):
    print(f"[{label}] CatBoost Optuna {N_TRIALS} trials...")
    study = optuna.create_study(direction="maximize")
    study.optimize(lambda t: obj_cat(t, X_tr, y_tr), n_trials=N_TRIALS, show_progress_bar=True)

    model = CatBoostClassifier(**study.best_params, random_seed=42, verbose=0)
    model.fit(X_tr, y_tr)

    prob  = model.predict_proba(X_te)[:,1]
    pred  = model.predict(X_te)
    tr_auc = roc_auc_score(y_tr, model.predict_proba(X_tr)[:,1])
    te_auc = roc_auc_score(y_te, prob)
    gap    = tr_auc - te_auc

    result = {
        "label":   label,
        "CV":      round(study.best_value, 4),
        "AUC":     round(te_auc, 4),
        "PR_AUC":  round(average_precision_score(y_te, prob), 4),
        "Acc":     round(accuracy_score(y_te, pred), 4),
        "F1":      round(f1_score(y_te, pred), 4),
        "gap":     round(gap, 4),
        "model":   model,
    }
    auc_te = result["AUC"]
    print(f"  CV={study.best_value:.4f}  Test AUC={auc_te:.4f}  Gap={gap:+.4f}")
    return result

print("Optuna 함수 정의 완료")

In [ ]:
result_base = run_cat(Xb_tr, yb_tr, Xb_te, yb_te, "기본 final_merged")
result_feat = run_cat(Xf_tr, yf_tr, Xf_te, yf_te, "파생변수 추가")

In [ ]:
print()
print("=" * 62)
print(" CatBoost Optuna 100 — 기본 vs 파생변수 추가 비교")
print("=" * 62)
fmt_h = "  {:<18}  {:>7}  {:>7}  {:>7}  {:>8}  {:>6}  {:>8}"
fmt_r = "  {:<18}  {:>7.4f}  {:>7.4f}  {:>7.4f}  {:>8.4f}  {:>6.4f}  {:>8}"
print(fmt_h.format("버전", "CV-AUC", "AUC", "PR-AUC", "Accuracy", "F1", "Gap"))
print("  " + "-" * 60)

for r in [result_base, result_feat]:
    flag = " << 과적합" if r["gap"] > 0.05 else ""
    print(fmt_r.format(r["label"], r["CV"], r["AUC"], r["PR_AUC"],
                       r["Acc"], r["F1"], f"{r['gap']:+.4f}{flag}"))

print("=" * 62)
diff = result_feat["AUC"] - result_base["AUC"]
print(f"\n파생변수 추가 효과: {diff:+.4f}")
if diff > 0.003:
    print("-> 파생변수가 유의미하게 기여함")
elif diff > 0:
    print("-> 파생변수가 소폭 기여함")
else:
    print("-> 파생변수 효과 없음 (기본 모델과 동일 수준)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False
from matplotlib.patches import Patch

derived = {"duration_days","plan_tier","currency_type","is_promotional_price",
           "amt_100","amt_7900","amt_10900","amt_13900","reg_weekday",
           "is_night_signup","is_same_day_cancel","age_group","has_watch_history",
           "avg_duration","watch_density","binge_score","recency","stream_watch_interaction"}

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, res, feats, title in [
    (axes[0], result_base, feat_base, "기본 final_merged"),
    (axes[1], result_feat, feat_feat, "파생변수 추가"),
]:
    fi = pd.Series(res["model"].feature_importances_, index=feats).sort_values(ascending=False)
    top20 = fi.head(20).sort_values()
    colors = ["#C00000" if f in derived else "#4472C4" for f in top20.index]
    top20.plot(kind="barh", ax=ax, color=colors[::-1])
    ax.set_title(f"Top 20 Feature Importance\n{title}\n빨강=파생변수 / 파랑=기본")
    ax.set_xlabel("Importance")

axes[1].legend(handles=[Patch(color="#C00000", label="파생변수"),
                         Patch(color="#4472C4", label="기본 피처")])
plt.tight_layout()
plt.show()

print("\n=== 파생변수 추가본 Top 15 ===")
fi_feat = pd.Series(result_feat["model"].feature_importances_, index=feat_feat).sort_values(ascending=False)
for i, (f, v) in enumerate(fi_feat.head(15).items(), 1):
    tag = " <- 파생" if f in derived else ""
    print(f"  {i:>2}. {f:<30} {v:.1f}{tag}")